# Model Evaluation for REFLECT
## Can these models see the scene? Can we trust their uncertainty?

REFLECT uses LLM logprobs over candidate actions as a *failure-prediction signal*.
That requires two things from the backbone:

1. **Perception that works on a real robot scene** - the model (or its perception
   front-end) must actually identify objects, their states, and spatial relations.
2. **Calibrated uncertainty** - the logprobs the planner reads must mean what
   they appear to mean, especially on the hard items where REFLECT actually needs
   to flag uncertainty.

This notebook tests both:

- **Part I** - qualitative sanity that a single holistic VLM caption is a usable
  perception substrate (3 AI2-THOR sim frames vs. ground-truth object lists).
- **Part II** - quantitative logprob calibration on **Robo2VLM-1**
  [Chen et al., NeurIPS 2025 Spotlight] across three perception conditions:
    * **A - VLM caption text** (standardized perception layer)
    * **B - native vision** (raw image to multimodal models)
    * **C - no perception** (choices only, sanity baseline)

Two design choices that matter:

- **No self-reported uncertainty.** Confidence comes from first-token logprobs over
  the option letters, not from asking the model how confident it is.
- **Dual difficulty axes.** Robo2VLM ships a `q_type` column (`spatial`, `success`,
  `action_direction`, `configuration`, `multiview`, `other`). We use it for
  category breakdowns and additionally derive a continuous per-item difficulty score
  from a **calibrator panel** of small models from families disjoint from the eval
  pool - giving a measured difficulty independent of question category.

## 0. Setup

In [ ]:
import os, math, base64, json, hashlib, io, re, random, time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional, List, Dict
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from datasets import load_dataset, load_from_disk
from ollama import Client

from reflect.llm.prompter import (
    LocalLLMPrompter,
    PortkeyLLMPrompter,
    score_choice_from_logprobs,
)

random.seed(42)
np.random.seed(42)
plt.rcParams.update({"figure.dpi": 110, "font.size": 10})

In [ ]:
# === Model registry ===
OLLAMA_MODELS = [
    "qwen3.5:9b", "qwen3.5:27b",           # text-only Qwen
    "qwen3-vl:8b", "qwen3-vl:30b",         # vision Qwen (separate model family)
    "deepseek-r1:14b", "gpt-oss:20b",      # text-only reasoning
    "gemma4:26b", "gemma4:31b",            # vision Gemma
]
OPENAI_MODELS = ["gpt-5.4"]

# qwen3.5:* is TEXT-ONLY - sending images to it makes ollama hang silently.
# Vision Qwen is qwen3-vl:*, a completely separate model.
MULTIMODAL = OPENAI_MODELS + [m for m in OLLAMA_MODELS
                               if "qwen3-vl" in m or "gemma4" in m]
TEXT_ONLY  = [m for m in OLLAMA_MODELS + OPENAI_MODELS if m not in MULTIMODAL]

# Must be a vision model. qwen3-vl:8b is fast and installed.
CAPTION_VLM = "qwen3-vl:8b"

# Calibrator panel: 5 small models from families DISJOINT from the eval pool
# (Meta / Mistral / Microsoft / IBM / Cohere - none overlap with
# Qwen / Gemma / DeepSeek / GPT-OSS / GPT-5.4). Used ONLY to derive empirical
# per-question difficulty; not evaluated or compared against the eval pool.
CALIBRATOR_MODELS = [
    "llama3.2:3b",    # Meta
    "mistral:7b",     # Mistral AI
    "phi4-mini:3.8b", # Microsoft
    "granite3.3:8b",  # IBM
    "command-r7b",    # Cohere
]

print(f"Eval pool (multimodal): {MULTIMODAL}")
print(f"Eval pool (text-only) : {TEXT_ONLY}")
print(f"Caption VLM           : {CAPTION_VLM}")
print(f"Calibrator panel      : {CALIBRATOR_MODELS}")

In [ ]:
# === Clients ===
PORTKEY_API_KEY = os.environ["PORTKEY_API_KEY"]  # set in your shell or .env (see .env.example)

portkey_client = PortkeyLLMPrompter(
    portkey_api_key=PORTKEY_API_KEY,
    model=OPENAI_MODELS[0],
    reasoning_effort="none",
)
# Explicit timeout on the ollama HTTP client so a stuck call surfaces as an
# error within OLLAMA_TIMEOUT_S instead of hanging the kernel forever.
OLLAMA_TIMEOUT_S = 120.0
ollama_client = Client(timeout=OLLAMA_TIMEOUT_S)

def make_local_prompter(model: str) -> LocalLLMPrompter:
    return LocalLLMPrompter(model_name=model)

print(f"Clients ready (ollama timeout = {OLLAMA_TIMEOUT_S:.0f}s).")

In [ ]:
# === Paths ===
from reflect.core.paths import sim_data_root, analysis_experiment_dir, analysis_cache_dir

SIM_DATA_ROOT = sim_data_root()                   # honors REFLECT_DATA_ROOT
ROBO2VLM_DIR  = analysis_cache_dir("robo2vlm_cache")
RESULTS_DIR   = analysis_experiment_dir("llm_calibration_robo2vlm")

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
ROBO2VLM_DIR.mkdir(parents=True, exist_ok=True)

for name, p in [("sim data", SIM_DATA_ROOT),
                 ("Robo2VLM cache", ROBO2VLM_DIR),
                 ("results dir", RESULTS_DIR)]:
    print(f"  [{'OK' if p.exists() else 'MISSING'}]  {name}: {p}")

In [ ]:
# === Holistic VLM caption helper ===
# One ollama call per image; 3-5 sentences listing visible objects, their
# states, and spatial relations. Perception bias from this single VLM is
# addressed by Condition C (no-perception sanity baseline) and the cross-VLM
# agreement check in Appendix A.
#
# Defensive measures (Robo2VLM images can be large and ollama silently hangs
# on oversize inputs):
#   - Resize to CAPTION_MAX_PX longest edge.
#   - keep_alive="10m" so the model stays loaded between calls (huge speed-up).
#   - Underlying ollama Client has a hard timeout (set in the clients cell).

CAPTION_PROMPT = (
    "Describe this robot manipulation scene in 3 to 5 sentences. "
    "List every visible object, its visible state (open/closed, on/off, "
    "full/empty, gripped/free), and its spatial relation to other objects "
    "(on top of, inside, next to, above/below, held by). Be exhaustive but "
    "terse. Do not interpret the task or guess intent. Output description only."
)
CAPTION_MAX_PX = 768   # longest-edge resize before sending to the VLM

def _to_caption_bytes(pil_image) -> bytes:
    img = pil_image.convert("RGB")
    w, h = img.size
    scale = CAPTION_MAX_PX / max(w, h)
    if scale < 1.0:
        img = img.resize((int(w * scale), int(h * scale)), Image.LANCZOS)
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return buf.getvalue()

def vlm_holistic_caption(pil_image, model: str = None) -> str:
    if pil_image is None:
        return ""
    model = model or CAPTION_VLM
    try:
        resp = ollama_client.chat(
            model=model,
            messages=[{"role": "user",
                       "content": CAPTION_PROMPT,
                       "images": [_to_caption_bytes(pil_image)]}],
            think=False,
            keep_alive="10m",
            options={"temperature": 0.0, "num_predict": 250},
        )
    except Exception as e:
        # Don't let one bad call abort the whole precompute loop.
        return f"[CAPTION_ERROR: {type(e).__name__}: {str(e)[:120]}]"
    msg  = resp["message"] if isinstance(resp, dict) else resp.message
    text = (msg.get("content") if isinstance(msg, dict) else msg.content) or ""
    if not text.strip():
        text = (msg.get("thinking") if isinstance(msg, dict)
                else getattr(msg, "thinking", "")) or ""
    return text.strip()

print("Holistic VLM caption helper ready.")

---
# Part I - Sanity check: does the captioner describe a robot scene?

Three AI2-THOR sim frames, captioned by `CAPTION_VLM` and compared against the
scene's ground-truth object list from `task.json`. This is *qualitative*; the
quantitative perception story is Part II's `acc_A` vs `acc_C` contrast.

If captions are obviously generic ("a kitchen scene") we stop here and pick a
different `CAPTION_VLM`. Otherwise we proceed: missing or hallucinated objects
in the caption will show up as miscalibration downstream, which is precisely
what Part II measures.

In [ ]:
TEST_FRAMES = [
    {"task": "boilWater",  "episode": "boilWater-1",  "step": 35, "label": "boilWater pre-failure"},
    {"task": "cookEgg",    "episode": "cookEgg-1",    "step": 33, "label": "cookEgg pre-failure"},
    {"task": "toastBread", "episode": "toastBread-1", "step": 42, "label": "toastBread pre-failure"},
]

for fr in TEST_FRAMES:
    img_path  = SIM_DATA_ROOT / fr["task"] / fr["episode"] / "ego_img" / f"img_step_{fr['step']}.png"
    task_path = SIM_DATA_ROOT / fr["task"] / fr["episode"] / "task.json"
    fr["path"]    = img_path
    fr["exists"]  = img_path.exists()
    fr["objects"] = []
    fr["gt_failure"] = None
    if task_path.exists():
        with open(task_path) as f:
            t = json.load(f)
        fr["objects"]    = t.get("object_list", [])
        fr["gt_failure"] = t.get("gt_failure_reason")
    flag = "OK" if fr["exists"] else "MISSING"
    print(f"  [{flag}]  {fr['label']:30s}  step={fr['step']}  n_objects={len(fr['objects'])}")

In [ ]:
# Caption + side-by-side display, with caching so reruns are cheap.
CAPTION_CACHE_I = RESULTS_DIR / "sim_frame_captions.json"
SIM_CAPTIONS    = json.load(open(CAPTION_CACHE_I)) if CAPTION_CACHE_I.exists() else {}

for fr in TEST_FRAMES:
    if fr["exists"] and fr["label"] not in SIM_CAPTIONS:
        t0 = time.time()
        SIM_CAPTIONS[fr["label"]] = vlm_holistic_caption(Image.open(fr["path"]))
        print(f"  {fr['label']:30s}  {time.time()-t0:.1f}s")

json.dump(SIM_CAPTIONS, open(CAPTION_CACHE_I, "w"), indent=2)

fig, axes = plt.subplots(len(TEST_FRAMES), 2,
                          figsize=(13, 3.5 * len(TEST_FRAMES)),
                          gridspec_kw={"width_ratios": [1, 1.8]})
for ax_row, fr in zip(axes, TEST_FRAMES):
    ax_img, ax_txt = ax_row
    if fr["exists"]:
        ax_img.imshow(Image.open(fr["path"]))
    ax_img.set_title(fr["label"], fontsize=10, fontweight="bold")
    ax_img.axis("off")
    cap = SIM_CAPTIONS.get(fr["label"], "(no caption)")
    obj = ", ".join(fr["objects"]) or "-"
    ax_txt.axis("off")
    ax_txt.text(0, 1,
                f"GT objects:\n  {obj}\n\nVLM caption:\n  {cap}",
                va="top", ha="left", fontsize=8.5, family="monospace", wrap=True)
plt.suptitle(f"Holistic VLM caption ({CAPTION_VLM}) vs AI2-THOR ground truth",
              fontweight="bold")
plt.tight_layout()
plt.show()

---
# Part II - Logprob calibration on Robo2VLM-1

The actual measurement. Each model in the evaluation pool is scored on a
stratified Robo2VLM sample across three perception conditions, and we
characterise calibration via ECE, entropy distributions, and split-conformal
coverage - sliced by empirical difficulty.

## II.1 Dataset

Chen et al., *"Robo2VLM: Visual Question Answering from Large-Scale In-the-Wild
Robot Manipulation Datasets"*, **NeurIPS 2025 Datasets & Benchmarks Track
(Spotlight)**. arXiv: 2505.15517. License: Apache 2.0.

- 684,710 five-way MCQ questions from 176K real robot trajectories (Open X-Embodiment).
- Ground truth derived from **physical sensors** (end-effector pose, gripper aperture,
  force sensing) - not from LM generation.
- Question types span spatial reasoning, goal-conditioned action prediction, and
  interaction/grasp reasoning - directly parallel to REFLECT planner decisions.

**Acknowledged limitation:** kitchen scenes are 16.9% of the dataset (office 33.6%,
lab 25.3%). Our downstream tasks are kitchen-focused, but the *reasoning types*
tested here (action selection, object state, spatial relations) transfer across
scene types - we are testing calibration, not kitchen visual knowledge.

In [ ]:
ROBO2VLM_HF_REPO    = "keplerccc/Robo2VLM-1"
SUBSET_PATH         = ROBO2VLM_DIR / "sampled_subset"
EPISODES_PER_TASK   = 5
SAMPLES_PER_EPISODE = 4

# Real schema (verified):
#   id, question, choices (str repr of Python list), correct_answer (int, 0-indexed),
#   image (PIL), task (str), episode (str), q_type (str)
# task / episode / q_type are native columns - no map() needed.

if SUBSET_PATH.exists():
    print("Loading precomputed subset from disk...")
    subset = load_from_disk(str(SUBSET_PATH))
else:
    print("Loading dataset from HF (uses cache if available)...")
    dataset = load_dataset(ROBO2VLM_HF_REPO, split="test",
                            cache_dir=ROBO2VLM_DIR / "hf_cache")

    print("Grouping episodes by task...")
    by_task = defaultdict(set)
    for ex in dataset:
        by_task[ex["task"]].add(ex["episode"])

    print("Sampling episodes per task...")
    selected = set()
    for task, episodes in by_task.items():
        selected.update(random.sample(list(episodes),
                                       min(EPISODES_PER_TASK, len(episodes))))

    print("Sampling questions within selected episodes...")
    by_ep = defaultdict(list)
    for i, ex in enumerate(dataset):
        if ex["episode"] in selected:
            by_ep[ex["episode"]].append(i)
    indices = []
    for _, idxs in by_ep.items():
        indices.extend(random.sample(idxs, min(SAMPLES_PER_EPISODE, len(idxs))))

    subset = dataset.select(indices)
    print(f"Subset size: {len(subset)}")
    subset.save_to_disk(str(SUBSET_PATH))

print("Columns :", subset.column_names)
print("Features:")
for k, v in subset.features.items():
    print(f"  {k:18s}  {v}")

from collections import Counter
print("\nq_type distribution:")
for k, v in sorted(Counter(ex["q_type"] for ex in subset).items(),
                   key=lambda x: -x[1]):
    print(f"  {k:25s} {v}")

SAMPLE_ITEMS = list(subset)
print(f"\n{len(SAMPLE_ITEMS)} questions ready.")
print(f"Example item[0]:")
print(f"  question       : {SAMPLE_ITEMS[0]['question'][:100]}")
print(f"  choices (raw)  : {SAMPLE_ITEMS[0]['choices'][:80]}")
print(f"  correct_answer : {SAMPLE_ITEMS[0]['correct_answer']}  (int, 0-indexed)")
print(f"  q_type         : {SAMPLE_ITEMS[0]['q_type']}")

## II.2 Evaluation protocol

For each `(model, condition, question)` triple we extract the first-token
logprob distribution over `A/B/C/D/E` (renormalised) and compute:

| field | meaning |
|---|---|
| `predicted` | argmax of the option distribution |
| `confidence` | `max(option_probs)` |
| `entropy` | normalised Shannon entropy of the option distribution, ∈ [0, 1] |
| `nonconformity` | `1 − P(correct answer)` - fed to split-conformal in II.10 |

**The three conditions:**

| condition | image? | scene-graph text? |
|---|---|---|
| **A - standardized** | no | VLM caption from `CAPTION_VLM` |
| **B - native vision** | yes (raw RGB) | no |
| **C - no perception** | no | no - choices only |

`C` is the sanity baseline: if `acc_A ≈ acc_C`, the question was text-solvable
from the choices alone and Condition A added nothing.

In [ ]:
import ast

@dataclass
class TaskResult:
    model:          str
    condition:      str           # A_standardized | B_native_vision | C_no_perception
    question_type:  str           # from item["q_type"]
    correct_answer: str           # single letter A-E
    predicted:      Optional[str]
    is_correct:     bool
    score_status:   str           # available | partial | text_fallback | missing_logprobs
    option_probs:   Dict[str, float] = field(default_factory=dict)
    confidence:     Optional[float] = None
    entropy:        Optional[float] = None
    nonconformity:  Optional[float] = None
    latency_s:      float = 0.0

    def to_dict(self) -> dict:
        return {**self.__dict__}


# --- Choice / answer parsing ---------------------------------------------
# Robo2VLM schema (verified):
#   choices       : str repr of a Python list, e.g. "['Yes', 'No', 'Cannot be determined']"
#                   Parse with ast.literal_eval.  Can have 4 OR 5 options.
#   correct_answer: int (0-indexed). Convert to letter: chr(ord('A') + int).
#   q_type        : native column ('spatial', 'success', 'action_direction', etc.)

def parse_choices(raw) -> Dict[str, str]:
    """Convert Robo2VLM choices field to {'A': '...', 'B': '...', ...}."""
    if isinstance(raw, dict):
        return {str(k).strip(): str(v).strip() for k, v in raw.items()}
    if isinstance(raw, list):
        return {chr(ord("A") + i): str(v).strip() for i, v in enumerate(raw)}
    if isinstance(raw, str):
        # Primary path: Python list repr e.g. "['Yes', 'No', ...]"
        try:
            lst = ast.literal_eval(raw)
            if isinstance(lst, list):
                return {chr(ord("A") + i): str(v).strip() for i, v in enumerate(lst)}
        except Exception:
            pass
        # Fallback: non-empty lines as options
        lines = [ln.strip() for ln in raw.splitlines() if ln.strip()]
        return {chr(ord("A") + i): ln for i, ln in enumerate(lines)}
    raise TypeError(f"Cannot parse choices of type {type(raw).__name__}: {raw!r}")

def normalize_answer(raw) -> str:
    """Convert Robo2VLM correct_answer (int, 0-indexed) to letter 'A'-'E'."""
    return chr(ord("A") + int(raw))

def build_prompt(question: str, choices: Dict[str, str], scene_ctx: str = "") -> str:
    ctx    = f"Scene context:\n{scene_ctx}\n\n" if scene_ctx else ""
    opts   = "\n".join(f"{k}: {v}" for k, v in choices.items())
    letters = "/".join(choices.keys())
    return f"{ctx}{question}\n\n{opts}\n\nAnswer with a single letter ({letters})."

def meta_to_result(model, condition, q_type, correct, meta, latency) -> TaskResult:
    pred  = meta.get("predicted_label")
    probs = meta.get("option_probs", {})
    nc    = 1.0 - probs.get(correct, 0.0) if probs else None
    return TaskResult(
        model=model, condition=condition, question_type=q_type,
        correct_answer=correct, predicted=pred, is_correct=(pred == correct),
        score_status=meta.get("score_status", "unknown"),
        option_probs=probs, confidence=meta.get("confidence"),
        entropy=meta.get("entropy"), nonconformity=nc, latency_s=latency,
    )

def is_text_only_condition(condition: str) -> bool:
    return condition in ("A_standardized", "C_no_perception")

print("Protocol defined.")

# Sanity check on first item.
_demo = SAMPLE_ITEMS[0]
_demo_choices = parse_choices(_demo["choices"])
_demo_answer  = normalize_answer(_demo["correct_answer"])
print(f"\nSanity check on item[0]:")
print(f"  question       : {_demo['question'][:100]}")
print(f"  choices parsed : {_demo_choices}")
print(f"  correct_answer : {_demo['correct_answer']} -> letter {_demo_answer!r}")
print(f"  correct text   : {_demo_choices.get(_demo_answer)!r}")
print(f"  q_type         : {_demo['q_type']}")

## II.3 Logprob probe

One probe per model to confirm logprobs are accessible and the `A-E`
distribution is non-degenerate (≥ 2 letters with non-zero mass). Models that
return `missing_logprobs` are eliminated: their calibration is unmeasurable
and they can't serve as a REFLECT backbone.

In [ ]:
PROBE_Q = ("A robot arm needs to grasp a cup on a table. The gripper is "
            "currently open and 15 cm above the cup. What should the robot do next?")
PROBE_C = {"A": "Close the gripper now",
            "B": "Move down then close gripper",
            "C": "Move to the left first",
            "D": "Open the gripper wider",
            "E": "Stop and request help"}
probe_prompt = build_prompt(PROBE_Q, PROBE_C)

def probe_ollama(model: str):
    t0 = time.time()
    _, meta = make_local_prompter(model).query(
        {"user": probe_prompt},
        {"max_tokens": 5, "temperature": 0.0},
        choice_spec=PROBE_C,
    )
    return time.time() - t0, meta

def probe_openai(model: str):
    t0 = time.time()
    resp = portkey_client._client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": probe_prompt}],
        logprobs=True, top_logprobs=5, max_completion_tokens=5,
    )
    msg  = resp.choices[0]
    text = msg.message.content.strip()
    lps  = getattr(msg.logprobs, "content", None)
    meta = score_choice_from_logprobs(lps, PROBE_C, response_text=text)
    return time.time() - t0, meta

PROBE_RESULTS = {}
print("Running logprob probe on the eval pool...\n")
for model in OLLAMA_MODELS + OPENAI_MODELS:
    try:
        latency, meta = (probe_openai if model in OPENAI_MODELS else probe_ollama)(model)
        meta  = meta or {}
        probs = meta.get("option_probs", {})
        n_nz  = sum(1 for p in probs.values() if p > 0)
        st    = meta.get("score_status", "no_meta")
        ok    = (st in ("available", "partial")) and (n_nz >= 2)
        PROBE_RESULTS[model] = {"ok": ok, "status": st, "n_nonzero": n_nz, "latency": latency}
        flag = "PASS" if ok else "FAIL"
        print(f"  [{flag}]  {model:20s}  status={st:18s}  nonzero={n_nz}  lat={latency:.1f}s")
    except Exception as e:
        PROBE_RESULTS[model] = {"ok": False, "status": "error", "error": str(e)}
        print(f"  [ERR]   {model:20s}  {str(e)[:80]}")

MODELS_PASSED_PROBE = [m for m, r in PROBE_RESULTS.items() if r.get("ok")]
print(f"\nCleared for full evaluation: {MODELS_PASSED_PROBE}")

## II.4 Pre-compute Condition A captions

One VLM call per Robo2VLM image; cached on disk so reruns are cheap.

In [ ]:
SCENE_CTX_CACHE = RESULTS_DIR / "robo2vlm_scene_ctx.json"

def img_key(item) -> str:
    img = item.get("image")
    if img is None:
        return f"noimg_{item['id']}"
    buf = io.BytesIO(); img.save(buf, format="PNG")
    return hashlib.sha1(buf.getvalue()).hexdigest()

SCENE_CTX = json.load(open(SCENE_CTX_CACHE)) if SCENE_CTX_CACHE.exists() else {}
needs    = [(qi, item) for qi, item in enumerate(SAMPLE_ITEMS) if img_key(item) not in SCENE_CTX]
print(f"Captioning {len(needs)} images via {CAPTION_VLM}  (cached: {len(SCENE_CTX)})")
print(f"Per-call timeout: {OLLAMA_TIMEOUT_S:.0f}s  |  resize cap: {CAPTION_MAX_PX}px")

# --- Pre-flight: caption one image and report timing. Surfaces config
#     problems (wrong model tag, model not pulled, ollama down) BEFORE the
#     real loop instead of leaving you wondering why the loop hangs.
if needs:
    qi0, item0 = needs[0]
    t0 = time.time()
    sample = vlm_holistic_caption(item0.get("image"))
    dt = time.time() - t0
    print(f"\nPre-flight on q{qi0}: {dt:.1f}s")
    print(f"  caption preview: {sample[:160]!r}")
    if sample.startswith("[CAPTION_ERROR"):
        raise RuntimeError(
            "Pre-flight caption failed. Most likely causes:\n"
            f"  - {CAPTION_VLM!r} not pulled (try: ollama pull {CAPTION_VLM})\n"
            "  - ollama daemon not running\n"
            "  - timeout too short for this model size"
        )
    SCENE_CTX[img_key(item0)] = sample
    json.dump(SCENE_CTX, open(SCENE_CTX_CACHE, "w"))
    needs = needs[1:]
    eta_total = dt * len(needs)
    print(f"  estimated remaining time: ~{eta_total/60:.1f} min for {len(needs)} items\n")

# --- Real loop with per-item progress, save-after-every-item, and skip-on-fail.
t_loop = time.time()
n_errors = 0
for k, (qi, item) in enumerate(needs, 1):
    t0 = time.time()
    cap = vlm_holistic_caption(item.get("image"))
    SCENE_CTX[img_key(item)] = cap
    json.dump(SCENE_CTX, open(SCENE_CTX_CACHE, "w"))
    if cap.startswith("[CAPTION_ERROR"):
        n_errors += 1
    elapsed_total = time.time() - t_loop
    eta = (elapsed_total / k) * (len(needs) - k)
    flag = "ERR" if cap.startswith("[CAPTION_ERROR") else "OK"
    print(f"  [{flag}]  q{qi:>4d}  {time.time()-t0:5.1f}s   "
           f"{k}/{len(needs)}   eta {eta/60:5.1f} min")

print(f"\nCaptions ready. Errors: {n_errors}/{len(needs)+1}")

## II.5 Main evaluation run

For each model in the eval pool × each condition × each question, score
logprobs. Results cached by `(model, q_idx, condition)`; reruns skip
already-completed work.

In [ ]:
def score_openai(item, model, condition):
    t0      = time.time()
    choices = parse_choices(item["choices"])
    answer  = normalize_answer(item["correct_answer"])
    q_type  = item.get("q_type", "")
    scene_ctx = SCENE_CTX.get(img_key(item), "") if condition == "A_standardized" else ""
    prompt_txt = build_prompt(item["question"], choices, scene_ctx)
    content = []
    if condition == "B_native_vision" and item.get("image") is not None:
        buf = io.BytesIO(); item["image"].save(buf, format="PNG")
        b64 = base64.b64encode(buf.getvalue()).decode()
        content.append({"type": "image_url",
                        "image_url": {"url": f"data:image/png;base64,{b64}"}})
    content.append({"type": "text", "text": prompt_txt})
    resp = portkey_client._client.chat.completions.create(
        model=model, messages=[{"role": "user", "content": content}],
        logprobs=True, top_logprobs=5, max_completion_tokens=5,
    )
    msg  = resp.choices[0]
    text = msg.message.content.strip()
    lps  = getattr(msg.logprobs, "content", None)
    meta = score_choice_from_logprobs(lps, choices, response_text=text)
    return meta_to_result(model, condition, q_type, answer, meta or {}, time.time() - t0)


def score_ollama(item, model, condition):
    t0      = time.time()
    choices = parse_choices(item["choices"])
    answer  = normalize_answer(item["correct_answer"])
    q_type  = item.get("q_type", "")
    use_text_only = (is_text_only_condition(condition)
                     or model in TEXT_ONLY
                     or item.get("image") is None)
    if use_text_only:
        scene_ctx = SCENE_CTX.get(img_key(item), "") if condition == "A_standardized" else ""
        prompt_txt = build_prompt(item["question"], choices, scene_ctx)
        _, meta = make_local_prompter(model).query(
            {"user": prompt_txt},
            {"max_tokens": 5, "temperature": 0.0},
            choice_spec=choices,
        )
    else:
        buf = io.BytesIO(); item["image"].save(buf, format="PNG")
        prompt_txt = build_prompt(item["question"], choices)
        resp = ollama_client.chat(
            model=model,
            messages=[{"role": "user", "content": prompt_txt,
                       "images": [buf.getvalue()]}],
            logprobs=True, top_logprobs=5,
            options={"num_predict": 5, "temperature": 0.0},
        )
        text = resp.message.content.strip()
        meta = score_choice_from_logprobs(
            getattr(resp, "logprobs", None), choices, response_text=text)
    return meta_to_result(model, condition, q_type, answer, meta or {}, time.time() - t0)


def score_one(item, model, condition):
    return (score_openai if model in OPENAI_MODELS else score_ollama)(item, model, condition)

print("Scoring helpers ready.")

In [ ]:
EVAL_CACHE   = RESULTS_DIR / "robo2vlm_eval_results.json"
ALL_RESULTS  = json.load(open(EVAL_CACHE)) if EVAL_CACHE.exists() else []
ALL_RESULTS  = [r for r in ALL_RESULTS if "condition" in r]   # drop legacy entries
done         = {(r["model"], r.get("q_idx"), r["condition"]) for r in ALL_RESULTS}

# EVAL_ITEMS is the working set for both the main eval and the calibrator panel.
# 500 items is enough for per-q_type and per-difficulty breakdowns while keeping
# the total call count manageable. Increase when doing a final sweep.
EVAL_ITEMS = random.sample(SAMPLE_ITEMS, min(500, len(SAMPLE_ITEMS)))
print(f"Eval set: {len(EVAL_ITEMS)} items  |  "
      f"q_type mix: {dict(Counter(it['q_type'] for it in EVAL_ITEMS))}")

to_run = []
for model in MODELS_PASSED_PROBE:
    conds = ["A_standardized", "C_no_perception"]
    if model in MULTIMODAL:
        conds.append("B_native_vision")
    for qi, item in enumerate(EVAL_ITEMS):
        for cond in conds:
            if (model, qi, cond) not in done:
                to_run.append((model, qi, cond, item))

print(f"Pairs to evaluate: {len(to_run)}  (cached: {len(done)})")

errors = 0
for model, qi, cond, item in to_run:
    try:
        res = score_one(item, model, cond)
        d   = res.to_dict(); d["q_idx"] = qi
        ALL_RESULTS.append(d)
        if len(ALL_RESULTS) % 50 == 0:
            json.dump(ALL_RESULTS, open(EVAL_CACHE, "w"), indent=2)
            print(f"  {len(ALL_RESULTS)} done  (last: {model} q{qi} {cond})")
    except Exception as e:
        errors += 1
        print(f"  [ERR] {model} q{qi} {cond}: {str(e)[:90]}")

json.dump(ALL_RESULTS, open(EVAL_CACHE, "w"), indent=2)
print(f"\nTotal results: {len(ALL_RESULTS)}  (errors this run: {errors})")

eval_df = pd.DataFrame(ALL_RESULTS)
print(eval_df.groupby(["model", "condition"])["is_correct"].agg(
    correct="sum", total="count", accuracy="mean").round(3).to_string())

## II.6 Calibrator panel - empirical per-item difficulty

Robo2VLM ships a `q_type` column we use in the accuracy table. But `q_type`
is a category, not a difficulty score - "spatial" questions vary from trivially
easy to very hard. For calibration analysis we need a **continuous** difficulty
signal per item.

The calibrator panel runs `EVAL_ITEMS` under Condition A with **5 small models
from families disjoint from the eval pool** (Meta, Mistral, Microsoft, IBM,
Cohere). Per-item difficulty is `1 − mean(is_correct)` across the panel.

Design rationale:
- **No self-reported uncertainty** - panel reports answers, not introspection.
- **No circularity** - families are disjoint, so difficulty isn't "what our
  eval models happen to find hard".
- **Complements q_type** - a spatial question with panel accuracy 1.0 is
  easy-spatial; one with accuracy 0.0 is hard-spatial. Both matter differently
  for calibration.

Native vision is not used in the panel: small-model vision quality varies too
much to be a stable difficulty instrument.

In [ ]:
PANEL_CACHE   = RESULTS_DIR / "robo2vlm_panel_results.json"
PANEL_RESULTS = json.load(open(PANEL_CACHE)) if PANEL_CACHE.exists() else []
panel_done    = {(r["model"], r["q_idx"]) for r in PANEL_RESULTS}

panel_to_run = [(m, qi, item)
                 for m in CALIBRATOR_MODELS
                 for qi, item in enumerate(EVAL_ITEMS)   # same set as the eval
                 if (m, qi) not in panel_done]
print(f"Panel evals to run: {len(panel_to_run)}")

panel_errors = 0
for model, qi, item in panel_to_run:
    try:
        res = score_ollama(item, model, "A_standardized")
        d   = res.to_dict(); d["q_idx"] = qi
        PANEL_RESULTS.append(d)
        if len(PANEL_RESULTS) % 50 == 0:
            json.dump(PANEL_RESULTS, open(PANEL_CACHE, "w"), indent=2)
    except Exception as e:
        panel_errors += 1
        print(f"  [ERR] {model} q{qi}: {str(e)[:90]}")

json.dump(PANEL_RESULTS, open(PANEL_CACHE, "w"), indent=2)
panel_df = pd.DataFrame(PANEL_RESULTS)
print(f"Panel total: {len(PANEL_RESULTS)} (errors this run: {panel_errors})")
print("\nPanel accuracy by model:")
print(panel_df.groupby("model")["is_correct"].agg(
    correct="sum", total="count", accuracy="mean").round(3).to_string())

### II.6.a Difficulty bucketing

Per question we keep two signals:
- `empirical_difficulty = 1 − mean(is_correct)` over the panel.
- `panel_js_div` = mean pairwise Jensen-Shannon divergence over the panel's
  option-probability distributions. Catches two failure modes:
  *uniformly-uncertain* (panel spreads mass evenly) vs.
  *confidently-disagreeing* (panel splits between specific wrong answers).

Items are then assigned to **easy / medium / hard** by tercile.

In [ ]:
def js_divergence(prob_lists):
    if len(prob_lists) < 2:
        return 0.0
    letters = sorted({k for p in prob_lists for k in p.keys()})
    if not letters:
        return 0.0
    def vec(p):
        v = np.array([p.get(k, 0.0) for k in letters]) + 1e-12
        return v / v.sum()
    def kl(a, b): return float((a * np.log2(a / b)).sum())
    pairs = []
    for i in range(len(prob_lists)):
        for j in range(i + 1, len(prob_lists)):
            a, b = vec(prob_lists[i]), vec(prob_lists[j])
            m = 0.5 * (a + b)
            pairs.append(0.5 * kl(a, m) + 0.5 * kl(b, m))
    return float(np.mean(pairs)) if pairs else 0.0


rows = []
for qi in range(len(EVAL_ITEMS)):    # only over the working set
    sub = panel_df[panel_df["q_idx"] == qi]
    if sub.empty:
        continue
    valid = sub[sub["score_status"].isin(["available", "partial"])]
    probs = [p for p in valid["option_probs"] if isinstance(p, dict) and p]
    rows.append({
        "q_idx":                qi,
        "panel_n":              len(sub),
        "panel_n_valid":        len(valid),
        "empirical_difficulty": float(1.0 - sub["is_correct"].mean()),
        "panel_js_div":         js_divergence(probs) if probs else 0.0,
    })
DIFFICULTY_DF = pd.DataFrame(rows).set_index("q_idx")
all_labels = ["easy", "medium", "hard"]
_, bins = pd.qcut(
    DIFFICULTY_DF["empirical_difficulty"],
    q=3, retbins=True, duplicates="drop"
)
n_bins = len(bins) - 1
labels = all_labels[-n_bins:]   # take last n: always ends with "hard"
DIFFICULTY_DF["difficulty_bucket"] = pd.qcut(
    DIFFICULTY_DF["empirical_difficulty"],
    q=3, labels=labels, duplicates="drop"
)


print(DIFFICULTY_DF.describe().round(3).to_string())
print("\nItems per bucket:")
print(DIFFICULTY_DF["difficulty_bucket"].value_counts().to_string())

# Merge difficulty + q_type into eval results.
# q_type comes from the item itself; difficulty from the panel.
q_type_map = {qi: EVAL_ITEMS[qi]["q_type"] for qi in range(len(EVAL_ITEMS))}
results_df  = eval_df.copy()
results_df["q_type_label"] = results_df["q_idx"].map(q_type_map)
results_df = results_df.merge(
    DIFFICULTY_DF[["empirical_difficulty", "panel_js_div", "difficulty_bucket"]],
    left_on="q_idx", right_index=True, how="left",
)
results_df["valid"] = results_df["score_status"].isin(["available", "partial"])
print("\nresults_df shape:", results_df.shape)

## II.7 Accuracy summary

In [ ]:
agg = results_df.groupby(["model", "condition"]).agg(
    accuracy     = ("is_correct",  "mean"),
    valid_rate   = ("valid",       "mean"),
    mean_entropy = ("entropy",     "mean"),
    mean_latency = ("latency_s",   "mean"),
    n            = ("is_correct",  "count"),
).round(3).sort_values("accuracy", ascending=False)
print("Per-(model, condition):")
print(agg.to_string())

# Accuracy by q_type (native Robo2VLM category).
print("\nAccuracy by q_type:")
qt_acc = results_df.groupby(
    ["model", "condition", "q_type_label"]
)["is_correct"].mean().unstack(fill_value=np.nan).round(2)
print(qt_acc.to_string())

# Accuracy by empirical-difficulty bucket.
print("\nAccuracy by difficulty bucket (panel-derived):")
diff_acc = results_df.groupby(
    ["model", "condition", "difficulty_bucket"], observed=True
)["is_correct"].mean().unstack(fill_value=np.nan).round(2)
print(diff_acc.to_string())

## II.8 Reliability & ECE

Expected Calibration Error: weighted average of `|confidence − accuracy|`
across confidence bins. ECE = 0 ⇔ perfectly calibrated.

Two views:
- **Aggregate** - one reliability diagram per `(model, condition)`.
- **Per difficulty bucket** - the deployment-relevant view. A model is a
  useful failure-prediction signal only if its calibration holds *on the
  hard items*. Aggregate ECE can be deceptively low when most items are easy.

In [ ]:
def compute_ece(confidences, is_correct, n_bins=10):
    if len(confidences) == 0:
        return float("nan")
    bins = np.linspace(0, 1, n_bins + 1)
    ece, n = 0.0, len(confidences)
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (confidences >= lo) & (confidences < hi)
        if mask.sum() == 0:
            continue
        ece += (mask.sum() / n) * abs(confidences[mask].mean() - is_correct[mask].mean())
    return ece

valid_df = results_df[results_df["valid"] & results_df["confidence"].notna()].copy()
groups   = list(valid_df.groupby(["model", "condition"]).groups.keys())

# === Aggregate reliability diagrams ===
ECE_SCORES = {}
ncols = min(3, len(groups)); nrows = math.ceil(len(groups) / max(ncols, 1))
fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 4*nrows), squeeze=False)

for ax_i, (model, cond) in enumerate(groups):
    sub  = valid_df[(valid_df["model"] == model) & (valid_df["condition"] == cond)]
    conf = sub["confidence"].values.astype(float)
    corr = sub["is_correct"].values.astype(float)
    ECE_SCORES[(model, cond)] = compute_ece(conf, corr)
    bins = np.linspace(0, 1, 11)
    bin_acc = [(corr[(conf>=lo)&(conf<hi)].mean()
                if ((conf>=lo)&(conf<hi)).sum() > 0 else np.nan)
                for lo, hi in zip(bins[:-1], bins[1:])]
    mid = (bins[:-1] + bins[1:]) / 2
    ax  = axes[ax_i // ncols][ax_i % ncols]
    ax.bar(mid, bin_acc, width=0.09, alpha=0.7, color="steelblue", label="Observed")
    ax.plot([0,1], [0,1], "k--", lw=1, label="Perfect")
    ax.set_xlim(0,1); ax.set_ylim(0,1)
    ax.set_xlabel("Confidence"); ax.set_ylabel("Accuracy")
    ax.set_title(f"{model}\n{cond} | ECE={ECE_SCORES[(model,cond)]:.3f}",
                  fontsize=9, fontweight="bold")
    ax.legend(fontsize=7)

for ax_i in range(len(groups), nrows * ncols):
    axes[ax_i // ncols][ax_i % ncols].axis("off")
plt.suptitle("Reliability diagrams - aggregate", fontweight="bold")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "reliability_aggregate.png", bbox_inches="tight", dpi=130)
plt.show()

print("Aggregate ECE:")
for k, v in ECE_SCORES.items():
    print(f"  {k[0]:20s} | {k[1]:18s}  {v:.4f}")

In [ ]:
# === Per-difficulty ECE - the REFLECT-relevant view ===
BUCKETS = ["easy", "medium", "hard"]
ECE_BY_DIFFICULTY = {}

for model, cond in groups:
    sub = valid_df[(valid_df["model"] == model) & (valid_df["condition"] == cond)]
    per_bucket = {}
    for b in BUCKETS:
        s = sub[sub["difficulty_bucket"] == b]
        per_bucket[b] = compute_ece(
            s["confidence"].values.astype(float),
            s["is_correct"].values.astype(float),
        ) if len(s) >= 5 else float("nan")
    ECE_BY_DIFFICULTY[(model, cond)] = per_bucket

ece_diff_df = pd.DataFrame(ECE_BY_DIFFICULTY).T.round(4)
ece_diff_df.index = pd.MultiIndex.from_tuples(ece_diff_df.index, names=["model", "condition"])
print("ECE by empirical difficulty:")
print(ece_diff_df.to_string())

fig, ax = plt.subplots(figsize=(max(8, len(groups)*1.4), 4.5))
x = np.arange(len(groups))
width = 0.27
for i, b in enumerate(BUCKETS):
    vals = [ECE_BY_DIFFICULTY[g].get(b, np.nan) for g in groups]
    ax.bar(x + (i-1)*width, vals, width, label=b)
ax.axhline(0.10, color="k", linestyle=":", lw=1, label="ECE = 0.10")
ax.set_xticks(x)
ax.set_xticklabels([f"{m}\n{c}" for m, c in groups],
                    rotation=20, ha="right", fontsize=8)
ax.set_ylabel("ECE")
ax.set_title("Calibration by empirical difficulty (lower is better)", fontweight="bold")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "reliability_by_difficulty.png", bbox_inches="tight", dpi=130)
plt.show()

## II.9 Entropy distributions

Does entropy rise when the model is wrong? That's the failure-prediction
signal REFLECT relies on. If the *correct* and *wrong* entropy histograms
overlap, entropy can't discriminate.

In [ ]:
ent_df     = valid_df[valid_df["entropy"].notna()].copy()
ent_groups = list(ent_df.groupby(["model", "condition"]).groups.keys())
ncols = min(3, len(ent_groups)); nrows = math.ceil(len(ent_groups) / max(ncols, 1))
fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 3.5*nrows), squeeze=False)

for ax_i, (model, cond) in enumerate(ent_groups):
    sub   = ent_df[(ent_df["model"] == model) & (ent_df["condition"] == cond)]
    corr  = sub[sub["is_correct"]]["entropy"].dropna()
    wrong = sub[~sub["is_correct"]]["entropy"].dropna()
    ax    = axes[ax_i // ncols][ax_i % ncols]
    bins  = np.linspace(0, 1, 21)
    ax.hist(corr,  bins=bins, alpha=0.6, color="seagreen",
             label=f"Correct (n={len(corr)})", density=True)
    ax.hist(wrong, bins=bins, alpha=0.6, color="tomato",
             label=f"Wrong (n={len(wrong)})",  density=True)
    if len(corr):  ax.axvline(corr.mean(),  color="seagreen", linestyle="--", lw=1.5)
    if len(wrong): ax.axvline(wrong.mean(), color="tomato",   linestyle="--", lw=1.5)
    ax.set_title(f"{model} | {cond}", fontsize=9, fontweight="bold")
    ax.set_xlabel("Normalised entropy"); ax.set_ylabel("Density")
    ax.legend(fontsize=7)

for ax_i in range(len(ent_groups), nrows * ncols):
    axes[ax_i // ncols][ax_i % ncols].axis("off")
plt.suptitle("Entropy: correct vs wrong", fontweight="bold")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "entropy_distributions.png", bbox_inches="tight", dpi=130)
plt.show()

## II.10 Conformal prediction

Split-conformal at `α = 0.10` (target 90% coverage). Nonconformity score is
`1 − P(correct)`. Per `(model, condition)`:

| metric | meaning |
|---|---|
| `q_hat` | calibrated threshold on the nonconformity score |
| `coverage` | fraction of test items whose correct answer makes the prediction set |
| `avg_set_size` | mean number of choices kept (smaller = sharper) |

In [ ]:
ALPHA = 0.10

def conformal_eval(sub_df, alpha=0.10):
    nc = sub_df["nonconformity"].dropna().values
    if len(nc) < 20:
        return {"q_hat": None, "coverage": None, "avg_set_size": None, "n": len(nc)}
    n_cal = len(nc) // 2
    idx   = np.random.permutation(len(nc))
    cal_nc, test_nc = nc[idx[:n_cal]], nc[idx[n_cal:]]
    q_level = np.ceil((1 - alpha) * (n_cal + 1)) / n_cal
    q_hat   = float(np.quantile(cal_nc, min(q_level, 1.0)))
    coverage = float((test_nc <= q_hat).mean())
    probs_iter = sub_df["option_probs"].dropna().iloc[: min(50, len(sub_df))]
    sizes = [sum(1 for p in pr.values() if (1 - p) <= q_hat) for pr in probs_iter]
    avg   = float(np.mean(sizes)) if sizes else None
    return {"q_hat": round(q_hat, 4),
            "coverage": round(coverage, 4),
            "avg_set_size": round(avg, 2) if avg else None,
            "n": len(nc)}

conf_df = valid_df[valid_df["nonconformity"].notna()].copy()
if conf_df["option_probs"].dtype == object:
    conf_df["option_probs"] = conf_df["option_probs"].apply(
        lambda x: json.loads(x) if isinstance(x, str) else x)

CONFORMAL_RESULTS = {(m, c): conformal_eval(sub, ALPHA)
                      for (m, c), sub in conf_df.groupby(["model", "condition"])}

conf_table = pd.DataFrame({f"{m}|{c}": r for (m, c), r in CONFORMAL_RESULTS.items()}).T
print(f"Split-conformal at alpha = {ALPHA}  (target coverage = {1-ALPHA:.0%})\n")
print(conf_table.to_string())

groups_c = [(m, c) for (m, c), r in CONFORMAL_RESULTS.items() if r["coverage"] is not None]
labels   = [f"{m}\n{c}" for m, c in groups_c]
cov      = [CONFORMAL_RESULTS[k]["coverage"]    for k in groups_c]
sizes    = [CONFORMAL_RESULTS[k]["avg_set_size"] or 0 for k in groups_c]

x = np.arange(len(groups_c))
fig, ax1 = plt.subplots(figsize=(max(7, len(groups_c)*1.3), 4))
ax2 = ax1.twinx()
ax1.bar(x - 0.2, cov,   0.35, color="steelblue",  alpha=0.8, label="Coverage")
ax2.bar(x + 0.2, sizes, 0.35, color="darkorange", alpha=0.8, label="Avg set size")
ax1.axhline(1 - ALPHA, color="steelblue", linestyle="--", lw=1.5,
             label=f"Target {1-ALPHA:.0%}")
ax1.set_xticks(x); ax1.set_xticklabels(labels, rotation=20, ha="right", fontsize=8)
ax1.set_ylabel("Coverage", color="steelblue")
ax2.set_ylabel("Avg set size", color="darkorange")
ax1.set_ylim(0, 1.1)
ln1, lb1 = ax1.get_legend_handles_labels()
ln2, lb2 = ax2.get_legend_handles_labels()
ax1.legend(ln1 + ln2, lb1 + lb2, fontsize=8)
plt.title("Conformal: coverage vs set size", fontweight="bold")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "conformal.png", bbox_inches="tight", dpi=130)
plt.show()

## II.11 Temperature scaling

Post-hoc calibration for models with raw `ECE > 0.10`. We grid-search `T*` on a
held-out calibration split (minimising NLL) and recompute ECE on the rest.
After scaling the reliability diagrams should be flatter and ECE smaller.

In [ ]:
ECE_THRESHOLD = 0.10

def scale_probs(probs, T):
    letters = list(probs.keys())
    logits  = [math.log(max(probs[k], 1e-12)) / T for k in letters]
    m       = max(logits)
    e       = [math.exp(l - m) for l in logits]
    s       = sum(e)
    return {k: ex / s for k, ex in zip(letters, e)}

def nll(probs_list, corrects, T):
    return -sum(math.log(max(scale_probs(p, T).get(c, 1e-12), 1e-12))
                 for p, c in zip(probs_list, corrects)) / len(probs_list)

def find_temperature(sub_df):
    T_grid = np.concatenate([np.linspace(0.1, 2.0, 40), np.linspace(2.0, 5.0, 10)])
    rows = sub_df[sub_df["option_probs"].notna() & sub_df["confidence"].notna()]
    if len(rows) < 10:
        return 1.0
    n_cal = len(rows) // 2
    probs_list = rows.iloc[:n_cal]["option_probs"].tolist()
    corrects   = rows.iloc[:n_cal]["correct_answer"].tolist()
    return float(min(T_grid, key=lambda T: nll(probs_list, corrects, T)))

groups_overconf = [k for k, ece in ECE_SCORES.items() if ece > ECE_THRESHOLD]
print(f"(model, condition) needing calibration: {groups_overconf}\n")

T_STAR, ECE_AFTER = {}, {}
for key in groups_overconf:
    model, cond = key
    sub = valid_df[(valid_df["model"] == model) & (valid_df["condition"] == cond)].copy()
    if sub["option_probs"].dtype == object:
        sub["option_probs"] = sub["option_probs"].apply(
            lambda x: json.loads(x) if isinstance(x, str) else x)
    T = find_temperature(sub)
    T_STAR[key] = round(T, 3)
    sub["conf_scaled"] = sub.apply(
        lambda r: max(scale_probs(r["option_probs"], T).values())
                   if isinstance(r["option_probs"], dict) else r["confidence"],
        axis=1,
    )
    ECE_AFTER[key] = round(compute_ece(sub["conf_scaled"].values,
                                         sub["is_correct"].values), 4)
    print(f"  {model:20s} {cond:18s}  T*={T:.3f}  ECE  {ECE_SCORES[key]:.4f} -> {ECE_AFTER[key]:.4f}")

# Default T*=1.0 and ECE_after=ECE_raw for everything else.
for key in valid_df.groupby(["model", "condition"]).groups.keys():
    T_STAR.setdefault(key, 1.0)
    ECE_AFTER.setdefault(key, ECE_SCORES.get(key))

---
# Part III - Verdict

Combine accuracy, ECE, and conformal coverage into a single per-model decision.

## III.1 Per-condition pass/fail

Pass criteria:
- `logprob_valid` - passed the II.3 probe
- `ECE_final ≤ 0.10` - after temperature scaling
- `conformal_cov ≥ 0.85` - coverage at `α = 0.10`

In [ ]:
rows = []
for (model, cond), n_acc in agg.iterrows():
    probe_ok = PROBE_RESULTS.get(model, {}).get("ok", False)
    ece      = ECE_AFTER.get((model, cond), ECE_SCORES.get((model, cond)))
    cov      = CONFORMAL_RESULTS.get((model, cond), {}).get("coverage")
    avg      = CONFORMAL_RESULTS.get((model, cond), {}).get("avg_set_size")
    T        = T_STAR.get((model, cond), 1.0)
    acc      = float(n_acc["accuracy"])
    ece_ok   = (ece is not None) and (ece <= ECE_THRESHOLD)
    cov_ok   = (cov is not None) and (cov >= 0.85)
    rows.append({
        "model":         model,
        "condition":     cond,
        "logprob_valid": "PASS" if probe_ok else "FAIL",
        "accuracy":      f"{acc:.3f}",
        "ECE_final":     f"{ece:.4f}" if ece is not None else "-",
        "T_star":        T,
        "conformal_cov": f"{cov:.3f}" if cov is not None else "-",
        "avg_set_size":  f"{avg:.2f}" if avg is not None else "-",
        "verdict":       "PASS" if (probe_ok and ece_ok and cov_ok) else "FAIL",
    })

verdict_df = pd.DataFrame(rows).set_index(["model", "condition"])
print(verdict_df.to_string())

# REFLECT cares about reasoning calibration with a standardized perception
# layer (Condition A). That's the gate.
PASSED_UNDER_A = [m for (m, c), v in zip(verdict_df.index, verdict_df["verdict"])
                   if c == "A_standardized" and v == "PASS"]
print(f"\nPassed Condition A: {PASSED_UNDER_A}")

## III.2 Combined scorecard

`A` - VLM caption (standardized perception)
`B` - native vision (multimodal models only)
`C` - no perception (sanity baseline)

Deltas:
- `dECE_BA > 0` → native vision is *worse* calibrated than the caption text.
  The model's own vision encoder hurts more than it helps.
- `dECE_AC < 0` → adding the caption *improves* calibration vs. choices-only.
  The model actually uses perception rather than guessing from the answer set.

In [ ]:
def cell(model, cond, key):
    return verdict_df.loc[(model, cond), key] if (model, cond) in verdict_df.index else "-"

all_models = list(OLLAMA_MODELS) + OPENAI_MODELS
score_rows = []
for model in all_models:
    has_A = (model, "A_standardized")  in verdict_df.index
    has_B = (model, "B_native_vision") in verdict_df.index
    has_C = (model, "C_no_perception") in verdict_df.index
    ece_A = ECE_AFTER.get((model, "A_standardized"))  or ECE_SCORES.get((model, "A_standardized"))
    ece_B = ECE_AFTER.get((model, "B_native_vision")) or ECE_SCORES.get((model, "B_native_vision"))
    ece_C = ECE_AFTER.get((model, "C_no_perception")) or ECE_SCORES.get((model, "C_no_perception"))
    d_BA  = (ece_B - ece_A) if (ece_A is not None and ece_B is not None) else None
    d_AC  = (ece_A - ece_C) if (ece_A is not None and ece_C is not None) else None
    score_rows.append({
        "model":         model,
        "logprob_valid": "PASS" if PROBE_RESULTS.get(model, {}).get("ok") else "FAIL",
        "acc_A":   cell(model, "A_standardized",  "accuracy") if has_A else "-",
        "ECE_A":   cell(model, "A_standardized",  "ECE_final") if has_A else "-",
        "cov_A":   cell(model, "A_standardized",  "conformal_cov") if has_A else "-",
        "acc_B":   cell(model, "B_native_vision", "accuracy") if has_B else "-",
        "ECE_B":   cell(model, "B_native_vision", "ECE_final") if has_B else "-",
        "cov_B":   cell(model, "B_native_vision", "conformal_cov") if has_B else "-",
        "acc_C":   cell(model, "C_no_perception", "accuracy") if has_C else "-",
        "ECE_C":   cell(model, "C_no_perception", "ECE_final") if has_C else "-",
        "dECE_BA": f"{d_BA:+.4f}" if d_BA is not None else "-",
        "dECE_AC": f"{d_AC:+.4f}" if d_AC is not None else "-",
        "verdict": "PASS" if model in PASSED_UNDER_A else "FAIL",
    })

scorecard = pd.DataFrame(score_rows).set_index("model")
print("\n=== REFLECT Model Evaluation - Scorecard ===")
print(scorecard.to_string())
scorecard.to_csv(RESULTS_DIR / "scorecard.csv")
print(f"\nSaved -> {RESULTS_DIR / 'scorecard.csv'}")

## III.3 Selection

Condition A isolates **reasoning calibration** from raw-vision quality.
REFLECT's perception layer is swappable; the LLM's intrinsic uncertainty
calibration is not. So we select on Condition A and sanity-check with B and C.

The empirical-difficulty slice (II.8 second panel) is the deployment-relevant
view: a model that is well-calibrated in aggregate but miscalibrated on the
*hard* items will silently fail at exactly the moments REFLECT needs it to
flag uncertainty.

*Fill in after running end-to-end:*

> Model `<X>` passes Condition A (acc =..., ECE =..., conformal cov =... at α = 0.10).
> `dECE_BA` =..., `dECE_AC` =... → perception bottleneck is manageable /
> problematic for deployment. ECE_hard =... (cf. ECE_easy =...) confirms the
> model is / is not still calibrated where calibration matters most.
> Selected as the REFLECT backbone.

---
## Appendix A - Cross-VLM caption variance (perception bias check)

Optional. Run if you want to directly quantify how much of the Condition-A
story is driven by `CAPTION_VLM` specifically vs. the downstream reasoning LLM.

For a subsample of Robo2VLM images we re-caption with one or more alternative
VLMs, compute per-image token-Jaccard agreement between the resulting captions,
and correlate that agreement with downstream Condition-A correctness:

- **ρ > 0** → low-agreement images cause more downstream errors. Perception is
  the bottleneck; caption choice matters.
- **ρ ≈ 0** → reasoning layer is robust to caption disagreement. Single-VLM
  perception bias is not driving the calibration story.

In [ ]:
ALT_CAPTION_VLMS = ["gemma4:26b"]    # add more (e.g. "gpt-5.4") to extend
CAPTION_VLM     = "qwen3.5:9b"
XVL_SUBSAMPLE    = min(60, len(SAMPLE_ITEMS))
XVL_CACHE        = RESULTS_DIR / "xvl_caption_variance.json"
XVL              = json.load(open(XVL_CACHE)) if XVL_CACHE.exists() else {}

def _tokens(t: str) -> set:
    t = re.sub(r"[^a-z0-9\s]", " ", (t or "").lower())
    return {w for w in t.split() if len(w) > 2}

def jaccard(a: str, b: str) -> float:
    A, B = _tokens(a), _tokens(b)
    if not A and not B:
        return 1.0
    return len(A & B) / max(1, len(A | B))

print(f"Captioning {XVL_SUBSAMPLE} items with: {[CAPTION_VLM] + ALT_CAPTION_VLMS}")
for qi in range(XVL_SUBSAMPLE):
    item = SAMPLE_ITEMS[qi]
    key  = img_key(item)
    XVL.setdefault(key, {})
    XVL[key].setdefault(CAPTION_VLM, SCENE_CTX.get(key, ""))
    for vlm in ALT_CAPTION_VLMS:
        if XVL[key].get(vlm):
            continue
        try:
            XVL[key][vlm] = vlm_holistic_caption(item.get("image"), model=vlm)
        except Exception as e:
            print(f"  [ERR] {vlm} q{qi}: {str(e)[:80]}")
            XVL[key][vlm] = ""
json.dump(XVL, open(XVL_CACHE, "w"))

xvl_rows = []
for qi in range(XVL_SUBSAMPLE):
    caps  = XVL.get(img_key(SAMPLE_ITEMS[qi]), {})
    vlms  = list(caps.keys())
    pairs = [jaccard(caps[vlms[i]], caps[vlms[j]])
              for i in range(len(vlms))
              for j in range(i + 1, len(vlms))]
    xvl_rows.append({
        "q_idx":             qi,
        "caption_agreement": float(np.mean(pairs)) if pairs else float("nan"),
        "n_vlms":            len(vlms),
    })
xvl_df = pd.DataFrame(xvl_rows).set_index("q_idx")

print(f"\nMean caption agreement (Jaccard): {xvl_df['caption_agreement'].mean():.3f}  "
       f"(n={len(xvl_df)})")

condA = results_df[results_df["condition"] == "A_standardized"].merge(
    xvl_df, left_on="q_idx", right_index=True, how="inner")
print("\nSpearman rho(caption_agreement, is_correct) per model:")
from scipy.stats import spearmanr
for model, sub in condA.groupby("model"):
    s = sub.dropna(subset=["caption_agreement", "is_correct"])
    if len(s) < 10:
        continue
    rho, p = spearmanr(s["caption_agreement"], s["is_correct"].astype(int))
    print(f"  {model:24s}  rho={rho:+.3f}  p={p:.3f}  n={len(s)}")